# M05 — Embedding、檢索與 RAG

本 notebook 對應 `README.md`，逐格執行即可。

我們會從「embedding 是什麼」的直覺開始，一路做到「用 LCEL 手刻一條
RAG 問答管線」。整條管線疊在 M03 學過的 `prompt | model | StrOutputParser()`
之上，差別只在前面多接了一段檢索。

## 1. 環境準備

載入共用 helper，取得「供應商無關」的 chat model 與 embeddings model。
`get_embeddings()` 是這個模組的新主角。

In [ ]:
# Load shared helpers so every notebook stays provider-agnostic.
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parents[1] / "_shared"))
from course_utils import get_model, get_embeddings, load_env

load_env()
model = get_model()
emb = get_embeddings()

## 2. Embedding 的直覺：文字 → 向量

`embed_query` 把一段文字壓成一個固定長度的數字陣列（向量）。
先看它的維度長怎樣。

Expected output（維度依模型而定，text-embedding-3-small 為 1536）：
向量維度： 1536
前 5 個數值： [0.012, -0.034, ...]

In [ ]:
vec = emb.embed_query("貓在沙發上睡覺")
print("向量維度：", len(vec))
print("前 5 個數值：", vec[:5])

## 3. 相似度的直覺：意思相近，向量也相近

我們用 cosine similarity（餘弦相似度）量化兩個向量「方向有多接近」，
值越接近 1 代表越相似。比較一句基準句對上「語意相近」與「語意無關」
兩句的分數，建立直覺。

注意：這裡手算只是為了教學透明，實務上向量庫會幫你做。

Expected output（數值僅示意，相對大小才是重點）：
相近句 相似度： 0.86
無關句 相似度： 0.12

In [ ]:
import math


def cosine_similarity(a, b):
    # Dot product divided by the product of magnitudes.
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    return dot / (norm_a * norm_b)


base = emb.embed_query("貓在沙發上睡覺")
similar = emb.embed_query("小貓正窩著打盹")
unrelated = emb.embed_query("今天台股大跌兩百點")

print("相近句 相似度：", round(cosine_similarity(base, similar), 3))
print("無關句 相似度：", round(cosine_similarity(base, unrelated), 3))

## 4. 切塊：把長文件切成小塊

文件不能整份丟進向量庫。用 `RecursiveCharacterTextSplitter` 沿著自然邊界
（段落 / 換行 / 句子）切塊。`chunk_size` 控制每塊大小，`chunk_overlap`
讓相鄰塊重疊一點，避免關鍵句剛好被切點劈斷。

這裡用一段關於 LangChain 的介紹文字當語料。

Expected output：
切出的塊數： 4
--- 第 1 塊 ---
LangChain 是一個...

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

long_text = """LangChain 是一個用來打造大型語言模型應用的框架。
它把「模型呼叫、prompt 模板、輸出解析、工具呼叫、檢索」這些常見零件標準化，
讓開發者可以像組積木一樣用 LCEL 的管線語法把它們串起來。

RAG（檢索增強生成）是 LangChain 最常見的應用之一。
它的核心想法是：先從外部知識庫檢索出與問題相關的段落，
再把這些段落連同問題一起交給模型，讓模型根據檢索到的內容作答。
這樣模型就能回答它原本沒被訓練過的私有或最新資料。

向量庫負責儲存文件的 embedding 並依相似度搜尋。
InMemoryVectorStore 把資料放在記憶體，適合教學與小型實驗；
正式環境通常會換成 Chroma 或 PGVector 等可持久化的方案，但操作介面相近。
"""

splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
chunks = splitter.split_text(long_text)

print("切出的塊數：", len(chunks))
for i, c in enumerate(chunks, 1):
    print(f"--- 第 {i} 塊 ---")
    print(c)

## 5. 建向量庫並加入文件

用 `InMemoryVectorStore` 綁定 embeddings model，再把切好的塊 `add_texts` 進去。
加入時每塊文字會被自動 embed 成向量存起來。

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

store = InMemoryVectorStore(emb)
store.add_texts(chunks)

## 6. 直接用 similarity_search 查詢

`similarity_search` 回傳最相似的 k 個 `Document`。先用它確認檢索抓到的
是不是我們要的段落。

Expected output：撈回的塊應與「RAG 是什麼」高度相關。
第 1 筆： RAG（檢索增強生成）是 LangChain 最常見的應用之一...

In [ ]:
hits = store.similarity_search("什麼是 RAG？", k=2)
for i, d in enumerate(hits, 1):
    print(f"第 {i} 筆：", d.page_content[:60].replace("\n", " "))

## 7. 包成 retriever（一個 Runnable）

`as_retriever()` 把搜尋包成一個可以 `.invoke()` 的 Runnable，
於是能像 M03 的任何組件一樣，用 `|` 串進 LCEL 管線。

Expected output：回傳一個 Document 清單，內容與查詢相關。

In [ ]:
retriever = store.as_retriever(search_kwargs={"k": 2})
docs = retriever.invoke("InMemoryVectorStore 適合什麼場景？")
for d in docs:
    print("-", d.page_content[:50].replace("\n", " "))

## 8. 手刻 RAG 管線（疊在 M03 LCEL 上）

資料流：問題 → 檢索文件 → format_docs 攤平成 context →
填進 prompt → model → StrOutputParser。

- retriever 給的是 Document 物件，prompt 要的是字串，所以中間用
  `format_docs` 把它們接起來。
- 字典寫法會「並行」準備 context 與 question 兩個 key（M03 學過）。
- `RunnablePassthrough()` 把原始問題字串原封不動傳給 question。

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "你是知識庫助理。只根據以下 context 回答問題，"
     "若 context 沒提到就回答「資料中沒有提到」，不要自行編造。\n\n"
     "context:\n{context}"),
    ("human", "{question}"),
])


def format_docs(docs):
    # Flatten retrieved Document objects into one plain-text block.
    return "\n\n".join(d.page_content for d in docs)


rag = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

## 9. 問問題

對著我們建好的知識庫提問。模型只會根據撈回的 context 作答。

Expected output（大意）：
RAG 會先從外部知識庫檢索與問題相關的段落，再連同問題交給模型作答...

In [ ]:
answer = rag.invoke("RAG 的核心想法是什麼？")
print(answer)

## 10. 測試「context 沒有的問題」

問一個語料裡沒有的問題，驗證 prompt 的防幻覺約束有沒有生效。

Expected output（大意）：資料中沒有提到。

In [ ]:
print(rag.invoke("LangChain 的作者是誰？"))

## 🧪 練習 1：換問題

用同一個 `rag` 管線，換 2~3 個不同問題試試（例如：
「向量庫負責什麼？」「正式環境會用什麼向量庫？」）。
觀察檢索撈回的段落是否切題，答案是否只引用 context。

In [ ]:
# Your code here. For example:
# print(rag.invoke("向量庫負責什麼？"))

## 🧪 練習 2：換語料

把第 4 格的 `long_text` 換成你自己的一段文字（例如一篇文章、
一份筆記，至少 3~4 段），重新跑第 4~7 格建立新的向量庫，
再用 `rag` 問與你語料相關的問題。

額外挑戰：把 `chunk_size` 從 120 調到 300，比較檢索回的塊有什麼不同，
體會 chunk 大小對檢索精準度的影響。

In [ ]:
# Your code here.

## 小結 & 下一步

你完成了一條完整的 RAG 管線：
切塊 → embed → 存入 InMemoryVectorStore → 用 retriever 檢索 →
format_docs 填進 prompt → model → 解析文字。

核心心法：**retriever 也只是一個 Runnable**，所以 RAG 不過是 M03
那條 LCEL 鏈往前多接一段檢索，沒有任何新魔法。

下一個模組 **M06（create_agent 與 Middleware）**：
目前流程都是我們寫死的固定管線。接下來讓模型自己決定要不要呼叫工具、
呼叫哪個、何時停止，把固定鏈升級成會自主決策的 Agent。